# Обратное распространение и минимальный цикл обучения

Авторы BelOAI. CC BY 4.0.

Выполняйте ячейки сверху вниз. Сначала решите задачи, затем обратитесь к решениям.

## Чему научимся

Прослеживать полный шаг обучения: прогноз, скалярная потеря, градиент и обновление.

## Четыре разные операции

[Autograd в Python](../../../python/pytorch-tools/autograd/) уже вычислял производные заданной формулы. Теперь добавляем обучаемую модель и обновление параметров: это превращает вычисление градиента в полный шаг обучения. Сам [градиент](../../../math/calculus/partial-derivatives/) сохраняет прежний математический смысл.

Сеть сначала вычисляет прогноз. Функция потерь сравнивает его с правильным ответом и сводит ошибки к скаляру. backward вычисляет производные по параметрам, а оптимизатор меняет параметры. Между шагами прежние градиенты нужно очищать: PyTorch по умолчанию накапливает их.

SGD — оптимизатор стохастического градиентного спуска. В нашем маленьком примере он получает сразу всю таблицу, поэтому шаг использует полный градиент. Параметр `lr` (learning rate) — размер шага η из [градиентного спуска](../../../math/optimization-methods/gradient-descent/): в простом SGD вес меняется на −η·градиент. zero_grad(set_to_none=True) очищает накопленное, step выполняет обновление. model.parameters() перечисляет обучаемые параметры.

## Разобранный пример: один вес

Пусть прогноз wx, x=2, y=6, w=1. Потеря (2w−6)²=16, градиент 2(2w−6)·2=−16. При η=0.1 новый вес 2.6, прогноз 5.2, потеря 0.64. backward сам не получил бы 2.6: он лишь записал −16 в градиент.

В сети правило цепочки проходит от потери к ранним слоям, используя сохранённые промежуточные значения. Это обратное распространение. Не нужно заново выводить формулу для каждого веса вручную, но маленький ручной пример помогает проверить знак и масштаб.

Проследим один цикл: при старых весах получаем прогноз → считаем одно число потери → `backward()` записывает градиенты → `step()` меняет веса. Сохранённое число потери относится к прогнозу **до** обновления. Чтобы узнать новую потерю, нужен новый прямой проход. В приведённом ниже коде вся таблица образует один пакет: каждая из 80 итераций — полный проход по обучающим объектам, то есть эпоха. Когда пакетов несколько, за эпоху выполняют несколько обновлений одной модели.

## Цикл обучения и проверка

В начале этапа model.train() задаёт режим обучения. Для оценки model.eval() и torch.no_grad() отдельно задают режим и отключают запись графа. Оценка не вызывает step. Если loss почти не меняется, проверьте формы, градиенты, размер шага и правильные ответы, прежде чем увеличивать сеть.

В примере учим линию y=2x на авторских точках и проверяем уменьшение ошибки. Это проверка механики, а не независимая оценка реальной задачи. Синтетический простой закон выбран специально, чтобы видеть каждый этап.

## Разобранный пример: проверяем вычисление

Все числа в примере придуманы для этого урока. Выполните код, затем измените вход и объясните новый результат.

In [ ]:
import torch
from torch import nn
torch.set_num_threads(1)
torch.manual_seed(17)
X = torch.tensor([[-1.], [0.], [1.], [2.]])
y = 2 * X
model = nn.Linear(1, 1)
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)
initial = ((model(X) - y)**2).mean().item()
for _ in range(80):
    optimizer.zero_grad(set_to_none=True)
    loss = ((model(X) - y)**2).mean()
    loss.backward()
    optimizer.step()
model.eval()
with torch.no_grad():
    final = ((model(X) - y)**2).mean().item()
assert final < initial * 0.01
print(initial, final)


## Типичные ошибки

Не забывайте очистку градиентов и step. Сравнивайте одинаковые формы y и прогнозов, чтобы broadcasting не создал таблицу ошибок.

## Задачи

### 1. Базовая

Меняет ли backward веса?

<details>
<summary>Подсказка</summary>

Он вычисляет производные.

</details>



### 2. Базовая

Зачем zero_grad?

<details>
<summary>Подсказка</summary>

Градиенты складываются.

</details>



### 3. Прикладная

Для ручного примера вычислите градиент при w=2.

<details>
<summary>Подсказка</summary>

2(2w−6)·2.

</details>



### 4. Прикладная

Если градиент −8 и η=0.1, как меняется w=2?

<details>
<summary>Подсказка</summary>

Вычесть отрицательное.

</details>



### 5. Олимпиадная
Предсказание — тензор с графом вычислений. Его заменили результатом `prediction.item()` и только из этого Python-числа построили потерю. Почему градиент не дойдёт до весов? Повредит ли отдельный вызов `prediction.item()` только для журнала, если потерю по-прежнему считать из исходного тензора?

<details>
<summary>Подсказка</summary>

Различайте создание копии числа для вывода и замену тензора на пути к потере.

</details>



### 6. Олимпиадная

Почему уменьшение обучающей ошибки не доказывает обобщение?

<details>
<summary>Подсказка</summary>

Измерены знакомые точки.

</details>



## Итоги и самопроверка

Назовите все четыре этапа и значение каждой переменной в ручном примере. Какие строки запрещены внутри итоговой оценки?

Для перехода дальше достаточно базовых и прикладных задач. Олимпиадные задачи можно отложить.


## Решения задач

### Задача 1

Нет. Обновление выполняет optimizer.step().

### Задача 2

Очистить результаты предыдущего шага.

### Задача 3

−8.

### Задача 4

До 2.8.

### Задача 5

Результат `item()` — обычное число без связи с графом. Если потеря вычислена только из него, путь к исходным весам утрачен: обычное число не поддерживает `backward`, а новый тензор из этого числа не восстановит прежний граф. Отдельный вызов `item()` для журнала исходный тензор не меняет. Если потеря продолжает использовать этот тензор, обратное распространение работает. Значит, важен путь от весов к потере, а не сам факт вызова `item()`.

### Задача 6

Нужна независимая проверка подходящих новых объектов.